In [ ]:

# Welcome to Machine Learning Housing Corp.! our task is to predict median house values in Californian districts, 
# given a number of features from these districts.


In [1]:
import sklearn
import numpy as np
import os
import tarfile
import urllib.request
import time


%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes',labelsize=14)
mpl.rc('xtick',labelsize=12)
mpl.rc("ytick",labelsize=12)



def fetch_housing_data(housing_url, housing_path, retries=5, timeout=10):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)

    tgz_path = os.path.join(housing_path, "housing.tgz")

    for attempt in range(1, retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading...")
            urllib.request.urlretrieve(housing_url, tgz_path)
            print("Download completed successfully.")
            break
        
        except Exception as e:
            print(f"Error: {e}")
            if attempt < retries:
                wait = attempt * 2
                print(f"Retrying in {wait} seconds...")
                time.sleep(wait)
            else:
                raise RuntimeError("Failed to download file after multiple attempts.")

    try:
        with tarfile.open(tgz_path) as housing_tgz:
            housing_tgz.extractall(path=housing_path)
        print("Extraction completed.")
    except Exception as e:
        raise RuntimeError(f"Failed to extract tgz file: {e}")


In [ ]:

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
Housing_Path = os.path.join("datasets", "housing")
Housing_Url = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

fetch_housing_data(Housing_Url, Housing_Path)

In [ ]:
import pandas as pd

def load_housing_data(housing_path=Housing_Path):
    csv_path = os.path.join(housing_path,'housing.csv')
    return pd.read_csv(csv_path)

In [6]:
# housing = load_housing_data()
# housing.head()

In [ ]:
housing.info()

In [ ]:
housing['ocean_proximity'].value_counts()

In [ ]:
housing.describe()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
housing.hist(bins=50,figsize=(20,15))
save_fig("attribute_histogram_plots")
plt.show()


In [ ]:
np.random.seed(6)

In [ ]:
import numpy as np

def split_train_test(data,test_ratio):
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data)*test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size]
    return data.iloc[train_indices],data.iloc[test_indices]

In [ ]:
train_set,test_set = split_train_test(housing,0.2)
len(train_set)

In [ ]:
len(test_set)

In [ ]:
from zlib import crc32

def test_set_check(identifier,test_ratio):
    return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32

def split_train_test_by_id(data,test_ratio,id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_ :test_set_check(id_,test_ratio))
    
    return data.iloc[~in_test_set], data.loc[in_test_set]


In [ ]:
import hashlib

def test_set_check(identifier,test_ratio,hash=hashlib.md5):
    return hash(np.int64(identifier).digest()[-1]<256*test_ratio)


In [ ]:
def test_set_check(identifier,test_ratio,hash=hashlib.md5):
    return bytearray(hash(np.int64(identifier)).digest())[-1] < 256 * test_ratio

In [ ]:
housing_with_id = housing.reset_index()
train_set,test_set = split_train_test_by_id(housing_with_id,0.2,'index')

In [ ]:
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "id")

In [ ]:
test_set.head()

In [ ]:
from sklearn.model_selection import train_test_split

trian_set,test_set = train_test_split(housing,test_size=0.2,random_state=6)

In [ ]:
test_set.head()

In [ ]:
housing['median_income'].hist()

In [ ]:
housing['income_cat'] = pd.cut(housing['median_income'],
                               bins=[0.,1.5,3.0,4.5,6.,np.inf],
                               labels=[1,2,3,4,5])

In [ ]:
housing['income_cat'].value_counts()

In [ ]:
housing['income_cat'].hist()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=6)
for train_index,test_index in split(housing,housing['income_cat']):
    start_train_set = housing.loc[train_index]
    start_test_set = housing.loc[test_index]

In [ ]:
start_test_set['income_cat'].value_counts()/len(start_test_set)

In [ ]:
def income_cat_proprtions(data):
    return data['income_cat'].value_counts()/len(data)

train_set,test_set = train_test_split(housing,test_size=0.2,random_state=6)

compare_props = pd.DataFrame({
    'Overall': income_cat_proprtions(housing),
    'Stratified':income_cat_proprtions(start_test_set),
    'Random':income_cat_proprtions(test_set),
}).sort_index()
compare_props["Rand. %error"] = 100 * compare_props["Random"] / compare_props["Overall"] - 100
compare_props["Strat. %error"] = 100 * compare_props["Stratified"] / compare_props["Overall"] - 100

In [ ]:
compare_props

In [ ]:
for set_ in (start_train_set,start_test_set):
    set_.drop('income_cat',axis=1,inplace=True)

In [ ]:
housing = start_train_set.copy()

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude")
save_fig("bad_visualization_plot")

In [ ]:
housing.plot(kind='scatter',x='longitude',y='latitude',alpha=0.1)
save_fig("bad_visualization_plot")

In [ ]:
housing.plot(kind='scatter',x='longitude',y='latitude',alpga=0.4,
                s=housing['population']/100,label='population',figsize=(10,7),
                c='median_house_value',cmap=plt.get_cmap('jet'),colorbar=True,
                sharex=False)
plt.legend()
save_fig('housing_prices_scatterplot')

In [ ]:
images_path = os.path.join(DOWNLOAD_ROOT, "images", "ML_PROJECT")
os.makedirs(images_path,exist_ok=True)
DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
filename = 'california.png'
print("Downloading", filename)
url = DOWNLOAD_ROOT + "images/end_to_end_project/" + filename
urllib.request.urlretrieve(url,os.path.join(images_path,filename))

In [ ]:
import matplotlib.image as mpimg
california_img = mpimg.imread(os.path.join(images_path,filename))
ax = housing.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
                        s=housing['population']/100,label='Population',
                        c='median_house_value',cmap=plt.get_cmap('jet'),
                        colorbar=False,alpha=0.4)
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,
           cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)

prices = housing['median_house_value']
tick_values = np.linspace(prices.min(),prices.max(),11)
cbar = plt.colorbar(ticks = tick_values/prices.max())
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value',fontsize=16)

plt.legend(fontsize=16)
save_fig('california_housing_prices_plot')
plt.show()

In [ ]:
corr_matrix = housing.corr(numeric_only=True)

In [ ]:
corr_matrix['median_house_value'].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix

attributes = ['median_house_value','median_income','total_rooms',
              'housing_median_age']
scatter_matrix(housing[attributes],figsize=(12,8))
save_fig('scatter_matrix_plot')

In [ ]:
housing.plot(kind='scatter',x='median_income',y='median_house_value',alpha=0.1)
plt.axis([0,16,0,55000])
save_fig('income_vc_house_value_scatterplot')

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

In [ ]:
corr_matrix = housing.corr()
corr_matrix['median_house_value'].sort_values(ascending=False)

In [ ]:
housing.plot(kind='scatter',x='rooms_per_household',y='median_house_value',alpha=0.2)
plt.axis([0,5,0,520000])
plt.show()

In [ ]:
housing.describe()

In [ ]:
housing = start_train_set.drop('median_house_value',axis=1)
housing_labels = start_train_set['median_house_value'].copy()

In [ ]:
#Data cleaning:


# housing.dropna(subset=['total_bedrooms'])  #option 1
# housing.drop('total_bedrooms',axis=1)   #option 2
# median =  housing['total_bedrooms'].median()  #option 3
# housing['total_bedrooms'].fillna(median,inplace=True)

In [ ]:
sample_incomplete_rows = housing[housing.isnull().any(axis=1)].head()
sample_incomplete_rows

In [ ]:
sample_incomplete_rows.dropna(subset=['total_bedrooms'])

In [ ]:
sample_incomplete_rows.dropna(subset=['total_bedrooms'])

In [ ]:
sample_incomplete_rows.drop('total_bedrooms',axis=1)

In [ ]:
median = housing['total_bedrooms'].median()
sample_incomplete_rows['total_bedrooms'].fillna(median,inplace=True)

In [ ]:
sample_incomplete_rows

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

In [ ]:
housing_num = housing.drop("ocean_proximity", axis=1)

In [ ]:
imputer.fit(housing_num)

In [ ]:
imputer.fit(housing_num)

In [ ]:
imputer.fit(housing_num)

In [ ]:
imputer.statistics_

In [ ]:
housing_num.median().values

In [ ]:
X = imputer.transform(housing_num)

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index=housing.index)

In [ ]:
housing_tr.loc[sample_incomplete_rows.index.values]

In [ ]:
imputer.strategy

In [ ]:
housing_tr = pd.DataFrame(X,columns=housing_num.columns,index=housing_num.index)

In [ ]:
housing_tr.head()

In [ ]:
housing_cat = housing[['ocean_proximity']]
housing_cat.head(10)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:10]

In [ ]:
ordinal_encoder.categories_

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

In [ ]:
housing_cat_1hot.toarray()

In [ ]:
cat_encoder = OneHotEncoder(sparse=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

In [ ]:
cat_encoder.categories_

In [ ]:
from sklearn.base import BaseEstimator,TransformerMixin

rooms_ix,bedrooms_ix,population_ix,households_ix = 3,4,5,6

class CombinedAttribureAdder(BaseEstimator,TransformerMixin):
    def __init__(self,add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self,X,y=None):
        return self
    def transform(self,X):
        rooms_per_household = X[:,rooms_ix]/X[:,households_ix]
        population_per_household = X[:,population_ix]/X[:,households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:,bedrooms_ix]/X[:,rooms_ix]
            return np.c_[X,rooms_per_household,population_per_household,bedrooms_per_room]
        else:
            return np.c_[X,rooms_per_household,population_per_household]

attr_adder = CombinedAttribureAdder(add_bedrooms_per_room=False)
housing_extra_attribs =  attr_adder.transform(housing.values)

In [ ]:
col_names = 'total_rooms','total_bedrooms','population','households'
rooms_ix,bedrooms_ix,population_ix,households_ix = [
    housing.columns.get_loc(c) for c in col_names
]

In [ ]:
housing_extra_attribs = pd.DataFrame(
    housing_extra_attribs,
    columns=list(housing.columns)+['rooms_per_household','population_per_household'],
    index=housing.index)
housing_extra_attribs.head()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('attribs_adder',CombinedAttribureAdder()),
    ('std_scaler',StandardScaler()),
])

housing_num_tr = num_pipeline.fit_transform(housing_num)

In [ ]:
housing_num_tr

In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs = list(housing_num)
cat_attribs = ['ocean_proximity']

full_pipeline = ColumnTransformer([
    ('num',num_pipeline,num_attribs),
    ('cat',OneHotEncoder(),cat_attribs)
])

housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
housing_prepared

In [ ]:
housing_prepared.shape

In [ ]:
from sklearn.base import BaseEstimator,TransformerMixin

class OldDataFrameSelector(BaseEstimator,TransformerMixin):
    def __init__(self,attribute_names):
        self.attribute_names = attribute_names
    def fit(self,X,y=None):
        return self
    def transform(self,X):
        return X[self.attribute_names].values

In [ ]:
num_attribs = list(housing_num)
cat_attribs = ['ocean_proximity']

old_num_pipeline = Pipeline([
    ('selector',OldDataFrameSelector(num_attribs)),
    ('imputer',SimpleImputer(strategy='median')),
    ('attribs_adder',CombinedAttribureAdder()),
    ('std_scaler',StandardScaler()),
])

old_cat_pipeline = Pipeline([
    ('selector',OldDataFrameSelector(cat_attribs)),
    ('cat_encoder',OneHotEncoder(sparse=False)),
])

In [ ]:
from sklearn.pipeline import FeatureUnion

old_full_pipeline = FeatureUnion(transformer_list={
    ('num_pipeline',old_num_pipeline),
    ('cat_pipeline',old_cat_pipeline),
})

In [ ]:
old_housing_prepared = old_full_pipeline.fit_transform(housing)
old_housing_prepared

In [ ]:
np.allclose(housing_prepared,old_housing_prepared)

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared,housing_labels)

In [ ]:
some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)


print('Predictions',lin_reg.predict(some_data_prepared))

In [ ]:
prices('Lables',list(some_labels))

In [ ]:
some_data_prepared

In [ ]:
from sklearn.metrics import mean_absolute_error
housing_predictions = lin_reg.predict(housing_prepared)
lin_mse = mean_absolute_error(housing_labels,housing_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse


In [ ]:
from sklearn.metrics import mean_absolute_error
lin_mae = mean_absolute_error(housing_labels,housing_predictions)

lin_mae

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=6)
tree_reg.fit(housing_prepared,housing_labels)

In [ ]:
housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_absolute_error(housing_labels,housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_mse

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg,housing_prepared,housing_labels,
                         scoring='neg_mean_squared_error',cv=10)
tree_rmse_scores = np.sqrt(-scores)

In [ ]:
def display_scores(scores):
    print('Scores',scores)
    print("Mean : ",scores.mean())
    print('Standard deviation: ', scores.std())
    
display_scores(tree_rmse_scores)

In [ ]:
lin_scores = cross_val_score(lin_reg,housing_prepared,housing_labels,
                             scoring='neg_mean_squared_error',cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_reg = RandomForestClassifier(n_estimators=100,random_state=6)
forest_reg.fit(housing_prepared,housing_labels)

In [ ]:
housing_predictions = forest_reg.predict(housing_prepared)
forest_mse = mean_squared_error(housing_labels, housing_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

In [ ]:
from sklearn.model_selection import cross_val_score
forest_scores = cross_val_score(forest_reg,housing_prepared,housing_labels,
                                scoring='neg_mean_squared_error',cv=10)

forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

In [ ]:
scores = cross_val_score(lin_reg,housing_prepared,housing_labels,scoring='neg_mean_squared_error',cv=10)
pr.Series(np.sqrt(-scores)).describe()

In [ ]:
from sklearn.svm import SVR
svm_reg = SVR(kernel='linear')
svm_reg.fit(housing_prepared,housing_labels)
housing_predictions = svm_reg.predict(housing_prepared)
svm_mse = mean_absolute_error(housing_labels,housing_predictions)
svm_rmse = np.sqrt(svm_mse)
svm_rmse

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'n_estimators':[3,10,30],'max_features':[2,4,6,8]},
    {'bootstrap':[False],'n_estimators':[3,10],'max_features':[2,3,4]},
]


forest_reg = RandomForestClassifier(random_state=6)

grid_search = GridSearchCV(forest_reg,param_grid,cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(housing_prepared,housing_labels)


In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_estimator_

In [ ]:
cvres = grid_search.cv_results_
for mean_score,params in zip(cvres['mean_test_score'],cvres['params']):
    prices(np.sqrt(-mean_score),params)

In [ ]:
pd.DataFrame(grid_search.cv_results_)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

params_distribs = {
    'n_estimators': randint(low=1,high=200),
    'max_features': randint(low=1,high=8),
}

forest_reg = RandomForestRegressor(random_state=42)

rnd_search = RandomizedSearchCV(forest_reg,param_distributions=params_distribs,
                                n_iter=10,cv=5,scoring='neg_mean_squared_error',random_state=6)

rnd_search.fit(housing_prepared,housing_labels)

In [ ]:
cvres = rnd_search.cv_results_
for mean_score,params  in zip(cvres['mean_test_score'],cvres['params']):
    print(np.sqrt(-mean_score),params)

In [ ]:
feature_importances = grid_search.best_estimator_.feature_importances_
feature_importances

In [ ]:
extra_attribs = ['rooms_per_hhold','pop_per_hhold','bedrooms_per_room']

cat_encoder = full_pipeline.named_transformers_['cat']
cat_one_hot_attribs = list(cat_encoder.categories_[0])
attributes = num_attribs + extra_attribs + cat_one_hot_attribs
sorted(zip(feature_importances,attributes),reverse=True)


In [ ]:
final_model = grid_search.best_estimator_

X_test = start_test_set.drop('median_house_value',axis=1)
y_test = start_test_set['median_house_value'].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)

final_mse = mean_squared_error(y_test,final_predictions)
final_rmse = np.sqrt(final_mse)

In [ ]:
final_rmse

In [ ]:
from scipy import stats

confidence = 0.95
squared_errors = (final_predictions-y_test)**2
np.sqrt(stats.t.interval(confidence,len(squared_errors)-1,
                         loc=squared_errors.mean(),
                         scale=stats.sem(squared_errors)))

In [ ]:
m = len(squared_errors)
mean = squared_errors.mean()
tscore = stats.t.ppf((1+confidence)/2,df=m-1)
tmargin = tscore*squared_errors.std(ddof=1)/np.sqrt(m)
np.sqrt(mean-tmargin),np.sqrt(mean+tmargin)

In [ ]:
zscore = stats.norm.ppf((1+confidence)/2)
zmargin = zscore*squared_errors.std(ddof=1)/np.sqrt(m)
np.sqrt(mean-zmargin),np.sqrt(mean+zmargin)

In [ ]:
full_pipeline_with_predictor = Pipeline([
    ('preparation',full_pipeline),
    ('linear',LinearRegression())
])

full_pipeline_with_predictor.fit(housing,housing_labels)
full_pipeline_with_predictor.predict(some_data)

In [ ]:
my_model = full_pipeline_with_predictor


In [ ]:
import joblib
joblib.dump(my_model,'my_model.pkl')

my_model_loaded = joblib.load('my_model.pkl')

In [ ]:
from scipy.stats import gemo,expon
gemo_distrib = gemo(0.5).rvs(10000,random_state=6)
expon_distrib = expon(scale=1).rvs(10000,random_state=6)
plt.hist(gemo_distrib,bins=50)
plt.show()
plt.hist(expon_distrib,bins=50)
plt.show()

**Warning**: the following cell may take close to 30 minutes to run, or more depending on your hardware.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'kernel':['linear'],'C':[10.,30.,100.,300.,1000.,3000.,10000.,30000.0]},
    {'kernel':['rbf'],'C':[1.0,3.0,10.,30.,100.,300.,1000.0],
     'gamma':[0.01,0.03,0.1,0.3,1.0,3.0]}
]

svm_reg = SVR()
grid_search = GridSearchCV(svm_reg,param_grid,cv=5,scoring='neg_mean_squared_error',verbose=2)
grid_search.fit(housing_prepared,housing_labels)

In [ ]:
negative_mse = grid_search.best_estimator_
rmse = np.sqrt(-negative_mse)
rmse

In [ ]:
grid_search.best_params_

**Warning**: the following cell may take close to 45 minutes to run, or more depending on your hardware.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon,reciprocal

param_distribs = {
    'kernel':['linear','rbf'],
    'C': reciprocal(20,200000),
    'gamma':expon(scale=1.0)
}

svm_reg = SVR()
rnd_search = RandomizedSearchCV(svm_reg,param_distributions=param_distribs,
                                n_iter=50,cv=5,scoring='neg_mean_squared_error',
                                verbose=2,random_state=6)

rnd_search.fit(housing_prepared,housing_labels)

In [ ]:
negative_mse = rnd_search.best_score_
rmse = np.sqrt(-negative_mse)
rmse

In [ ]:
rnd_search.best_params_

In [ ]:
expon_distrib = expon(scale=1.)
samples = expon_distrib.rvs(10000,random_state=6)
plt.figure(figsize=(10,4))
plt.subplot(121)
plt.title('Exponential distribution (scale=1.0)')
plt.hist(samples,bins=50)
plt.subplot(122)
plt.title('Log of this distribution')
plt.hist(np.log(samples),bins=50)
plt.show()

In [ ]:
reciprocal_distrib = reciprocal(20,20000)
samples = reciprocal_distrib.rvs(10000,random_state=6)
plt.figure(figsize=(10,4))
plt.subplot(121)
plt.title("Reciprocal distribution (scale=1.0)")
plt.hist(samples, bins=50)
plt.subplot(122)
plt.title('Log of this distriburion')
plt.hist(np.log(samples),bins=50)
plt.show()

In [ ]:

from sklearn.base import BaseEstimator,TransformerMixin

def indices_of_top_k(arr,k):
    return np.sort(np.argpartition(np.array(arr),-k)[-k:])

class TopFeatureSelector(BaseEstimator,TransformerMixin):
    def __init__(self,feature_imporatances,k):
        self.feature_imporatances = feature_imporatances
        self.k = k
    def fit(self,X,y=None):
        self.feature_indices_ = indices_of_top_k(self.feature_imporatances,self.k)
        return self
    def transform(self,X):
        return X[:,self.feature_indices_]

In [ ]:
k = 5

In [ ]:
top_k_feature_indices = indices_of_top_k(feature_importances,k)
top_k_feature_indices

In [ ]:
np.array(attributes)[top_k_feature_indices]

In [ ]:
sorted(zip(feature_importances,attributes),reverse=True)[:k]

In [ ]:
preparation_and_feature_selection_pipeline = Pipeline([
    ('prepartion',full_pipeline),
    ('feature_selection',TopFeatureSelector(feature_importances,k))
])

In [ ]:
housing_prepared_top_k_features = preparation_and_feature_selection_pipeline.fit_transform(housing)

In [ ]:
housing_prepared_top_k_features[:3]

In [ ]:
housing_prepared[0:3, top_k_feature_indices]

In [ ]:
prepare_select_and_predict_pipeline = Pipeline([
    ('prepartion',full_pipeline),
    ('feature_selection',TopFeatureSelector(feature_importances,k)),
    ('svm_reg',SVR(**rnd_search.best_estimator_))
])

In [ ]:
prepare_select_and_predict_pipeline.fit(housing,housing_labels)

In [ ]:
some_data = housing.iloc[:4]
some_labels = housing_labels.iloc[:4]



print("Predictions:\t", prepare_select_and_predict_pipeline.predict(some_data))
print("Labels:\t\t", list(some_labels))